In [0]:
from pyspark.sql import functions as F

S = lambda n: spark.table(f"workspace.silver.{n}")

def save(df, name):
    full = f"workspace.gold.{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full))
    print(f"{name}: {spark.table(full).count()} rows")

In [0]:
save(S("stores").select("store_id", "store_name", "city", "state", "opened_date"), "dim_store")

save(S("customers").select(
        "customer_id",
        F.concat_ws(" ", "first_name", "last_name").alias("customer_name"),
        "email", "phone", "city", "created_at"), "dim_customer")

save(S("products").select(
        "product_id", "title", "category", "price",
        "rating_rate", "rating_count"), "dim_product")

dim_store: 10 rows
dim_customer: 200 rows
dim_product: 20 rows


In [0]:
b = spark.sql("""
  SELECT min(d) AS mn, max(d) AS mx FROM (
    SELECT order_date AS d FROM workspace.silver.orders
    UNION ALL
    SELECT snapshot_date FROM workspace.silver.inventory)""").first()

dim_date = (spark.range(1)
    .select(F.explode(F.expr(f"sequence(to_date('{b.mn}'), to_date('{b.mx}'), interval 1 day)")).alias("date"))
    .select(
        F.date_format("date", "yyyyMMdd").cast("int").alias("date_key"),
        "date",
        F.year("date").alias("year"),
        F.quarter("date").alias("quarter"),
        F.month("date").alias("month"),
        F.date_format("date", "MMMM").alias("month_name"),
        F.dayofmonth("date").alias("day"),
        F.date_format("date", "EEEE").alias("day_name"),
        F.dayofweek("date").isin(1, 7).alias("is_weekend")))
save(dim_date, "dim_date")

dim_date: 88 rows


In [0]:
o, i = S("orders"), S("order_items")

fact_sales = (i.join(o, "order_id")
    .select(
        "order_item_id", "order_id",
        F.date_format("order_date", "yyyyMMdd").cast("int").alias("date_key"),
        "order_ts", "customer_id", "store_id", "product_id",
        "payment_mode", "order_status",
        "quantity", "unit_price", "line_total",
        (F.col("order_status") == "Completed").alias("is_completed"))
    .withColumn("net_revenue",
                F.when(F.col("is_completed"), F.col("line_total")).otherwise(F.lit(0).cast("decimal(12,2)"))))
save(fact_sales, "fact_sales")

fact_inv = (S("inventory").select(
        F.date_format("snapshot_date", "yyyyMMdd").cast("int").alias("date_key"),
        "store_id", "product_id", "stock_qty", "reorder_level",
        (F.col("stock_qty") < F.col("reorder_level")).alias("below_reorder_level")))
save(fact_inv, "fact_inventory_snapshot")

fact_sales: 12001 rows
fact_inventory_snapshot: 1400 rows


In [0]:
G = lambda n: spark.table(f"workspace.gold.{n}")

assert G("fact_sales").count() == S("order_items").count(), "fact_sales lost or gained rows"
assert G("fact_inventory_snapshot").count() == S("inventory").count(), "inventory fact row mismatch"

# Every fact key must exist in its dimension
for fact, dim, key in [("fact_sales","dim_customer","customer_id"),
                       ("fact_sales","dim_store","store_id"),
                       ("fact_sales","dim_product","product_id"),
                       ("fact_sales","dim_date","date_key"),
                       ("fact_inventory_snapshot","dim_store","store_id"),
                       ("fact_inventory_snapshot","dim_product","product_id"),
                       ("fact_inventory_snapshot","dim_date","date_key")]:
    bad = G(fact).join(G(dim), key, "left_anti").count()
    assert bad == 0, f"{fact}.{key}: {bad} rows without a match in {dim}"

print("Gold: all checks passed")
display(G("fact_sales").agg(F.sum("line_total").alias("gross"), F.sum("net_revenue").alias("net")))

Gold: all checks passed


gross,net
16407995.00,14111079.00
